# Echo Core

Echo Core runs a line of qubits through a scrambling circuit, nudges one qubit, then runs the same circuit backwards. Whatever fails to cancel is the echo: a map of taps, one per qubit and time step, each with a level and a sign. Some taps come back inverted, which no acoustic reflection can do; some never come back at all.

Echo Audio renders this map as a multi-tap delay on your sound. Here you look at the map itself.

## Setup

In [ ]:
import getpass
import os

import matplotlib.pyplot as plt
import numpy as np

from moth_atlas import Atlas

ENGINE = "otoc-echo-staging"  # the engine this notebook runs: Atlas V2 (staging). Production: "otoc-echo-v1"

if not os.environ.get("MOTH_API_KEY"):
    os.environ["MOTH_API_KEY"] = getpass.getpass("Atlas API key")

client = Atlas()  # reads MOTH_API_KEY, and MOTH_API_BASE for which API to call

In [ ]:
def tap_grid(result):
    """The tap map as a (depth, site) array of F's real part, from the result's flat tap list."""
    spec, taps = result["extras"]["spec"], result["extras"]["taps"]
    sites = spec["n_sites"] if spec["lattice"] == "chain" else spec["width"] * spec["height"]
    grid = np.zeros((spec["depth"], sites))
    for tap in taps:
        grid[tap["depth"] - 1, tap["site"]] = tap["F_re"]
    return grid


def tap_maps(results, size=2.6):
    """Draw {label: result} as tap maps side by side: red taps return the right way up, blue ones inverted, pale ones erased."""
    fig, axes = plt.subplots(1, len(results), figsize=(size * len(results), size + 0.6), squeeze=False)
    for ax, (label, result) in zip(axes[0], results.items()):
        grid = tap_grid(result)
        ax.imshow(grid, cmap="RdBu_r", vmin=-1, vmax=1, origin="lower", aspect="auto",
                  extent=(-0.5, grid.shape[1] - 0.5, 0.5, grid.shape[0] + 0.5))
        ax.axvline(result["extras"]["kick_site"], color="k", lw=0.5, ls=":")
        s = result["extras"]["summary"]
        ax.set_title(f"{label}\n{s['regular']} regular · {s['inverted']} inverted · {s['erased']} erased", fontsize=8)
        ax.set_xlabel("site (qubit)")
        ax.set_ylabel("depth (time slot)")
    plt.show()

## Try it

An 8-qubit chain, 8 time slots, kicked in the middle. The dotted line marks the kicked qubit; colour is each tap's value $F$, from +1 (red, a clean echo) through 0 (erased) to −1 (blue, inverted).

In [ ]:
job = client.jobs.run(
    engine=ENGINE,
    params={
        "n_sites": 8,
        "depth": 8,
        "theta_x": 0.94,
        "theta_zz": 1.1,
        "kick": "Z",
    },
)
tap_maps({"default drive": job.result})

## Parameters

What to expect from each:

- `theta_x` (0 to π): the transverse kick per layer, the "how quantum" dial. Low gives loud, regular taps; high gives inverted, attenuated and erased ones.
- `theta_zz` (−π to π): the coupling per layer. Non-monotonic: about 0.25π is sparse, π is a Clifford circuit that doesn't scramble at all.
- `theta_z` (0 to π): a phase layer. Above 0, taps become complex and carry a phase (`F_im`).
- `n_sites`, `depth`: the chain length (one site per stereo position) and the number of time slots.
- `kick`, `kick_site`: which Pauli disturbs the chain, and where (default: the centre).
- `disorder`, `seed`: random jitter on every gate angle, which localises the echo. `seed` only matters when `disorder` > 0.
- `lattice`, `width`, `height`: `square` uses a 2-D grid instead of a chain.
- `machine`: `aer` simulates; an IBM backend name such as `ibm_phoenix` measures on hardware. `exact` (aer only) uses exact expectation values instead of `shots`.
- `include_z`, `min_tap_level`, `twirls`, `fractional_gates`: extra readout, the tap-list threshold, and hardware error handling.

In [ ]:
# client.engines.get(ENGINE)  # its description, params, input and output files, from the API

## How it works

**Forward.** A line of $n$ qubits is driven by a Floquet layer: $R_{ZZ}(\theta_{zz})$ on every neighbouring pair, then $R_X(\theta_x)$ (and $R_Z(\theta_z)$) on every qubit. $t$ layers make $U_t$.

**Kick and reverse.** A single Pauli $P$ (the `kick`) hits one qubit, then the inverse circuit $U_t^\dagger$ runs gate for gate. Without the kick, everything would cancel and the chain would return to its start.

**Listen.** Each qubit is measured. Comparing it with a reference run without the kick gives one number per site and depth, $F$: this is an out-of-time-order correlator (OTOC), the standard probe of how fast a quantum system scrambles information.

$F \approx 1$ means the disturbance never reached that site, or came back intact. $F < 0$ means it returned inverted. $|F| \approx 0$ means it was scrambled beyond recovery. Information can only spread one site per layer, which is the light cone you can see fanning out from the kicked qubit.

As a delay line: site → stereo position, depth → time slot, $|F|$ → level, sign → polarity.

## Experiments

Each experiment changes one parameter and keeps the others fixed, and runs one job per value.

### How quantum (`theta_x`)

From a nearly classical chain, where the kick passes through cleanly, to a strongly scrambling one where most taps flip or vanish.

In [ ]:
results = {}
for theta_x in (0.3, 0.94, 1.6, 2.4):
    print(f"theta_x={theta_x}")
    results[f"theta_x {theta_x}"] = client.jobs.run(engine=ENGINE, params={"theta_x": theta_x}, quiet=True).result
tap_maps(results)

### Coupling (`theta_zz`)

The coupling is non-monotonic: about 0.25π gives a sparse map, the default about 0.35π a dense one, and π stops scrambling.

In [ ]:
results = {}
for theta_zz in (0.25 * np.pi, 0.35 * np.pi, 0.6 * np.pi, np.pi):
    print(f"theta_zz={theta_zz:.3f}")
    results[f"theta_zz {theta_zz / np.pi:.2f}π"] = client.jobs.run(engine=ENGINE, params={"theta_zz": theta_zz}, quiet=True).result
tap_maps(results)

### Disorder

Jitter on every gate angle stops the echo from spreading: the taps stay near the kicked qubit. Two seeds give two different realisations of the same disorder.

In [ ]:
results = {}
for disorder, seed in ((0.0, None), (0.5, 1), (0.5, 2), (1.0, 1)):
    params = {"disorder": disorder, **({"seed": seed} if seed is not None else {})}
    print(params)
    results[f"disorder {disorder}" + (f", seed {seed}" if seed else "")] = client.jobs.run(engine=ENGINE, params=params, quiet=True).result
tap_maps(results)

### Phase (`theta_z`)

With a phase layer, taps become complex. Here: the share of each tap's level in its imaginary part, by site and depth.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(8, 3))
for ax, theta_z in zip(axes, (0.0, 0.5, 1.2)):
    print(f"theta_z={theta_z}")
    result = client.jobs.run(engine=ENGINE, params={"theta_z": theta_z}, quiet=True).result
    spec = result["extras"]["spec"]
    share = np.zeros((spec["depth"], spec["n_sites"]))
    for tap in result["extras"]["taps"]:
        share[tap["depth"] - 1, tap["site"]] = abs(tap["F_im"]) / max(tap["level"], 1e-9)
    ax.imshow(share, cmap="viridis", vmin=0, vmax=1, origin="lower", aspect="auto",
              extent=(-0.5, share.shape[1] - 0.5, 0.5, share.shape[0] + 0.5))
    ax.set_title(f"theta_z {theta_z}", fontsize=9)
    ax.set_xlabel("site")
axes[0].set_ylabel("depth")
plt.show()

## Questions

**What is an OTOC?**

An out-of-time-order correlator: run a system forward, disturb it, run it backwards, and see what's left. It measures how quickly a disturbance spreads and becomes unrecoverable, and is used to study quantum chaos and information scrambling.

**How do I hear it?**

Echo Audio (`retrocausal_echo/`) turns this map into a multi-tap delay. Its `ir` input takes an Echo Core result, so one measurement can drive many mixes.

**Does it run on hardware?**

Set `machine` to an IBM backend name (your account needs QPU access) and use `shots` instead of `exact`. `twirls` and `fractional_gates` help against hardware error; expect queues of minutes to hours.

**Why is the light cone shaped like that?**

Each layer couples only neighbours, so the kick can reach at most one more site per layer. Sites outside the cone return $F = 1$: the disturbance never got there.